# Payment Auth, Clearing, and Settlement Lab

Additional payment analytics lab notebook included as a reviewable portfolio artifact.

This public portfolio copy keeps the full notebook source visible on GitHub while removing execution outputs, execution counts, and environment-specific metadata.

# Payment Analysis Lab — Auth, Clearing & Settlement
### MSc Data Science · Payment Analysis for Open Banking

This notebook uses three linked datasets that simulate a real payment processor's data pipeline across 30 days of transactions.

| File | Rows | Description |
|------|------|-------------|
| `auth_records.csv` | ~777 | Real-time authorisation events (approved, declined, timeout) |
| `clearing_records.csv` | ~645 | End-of-day cleared transactions with fee breakdown |
| `settlement_records.csv` | ~491 | Daily net settlement positions by acquirer-issuer pair |

**Key column to join on:** `auth_id` links auth → clearing. `clearing_id` links clearing → settlement (via `settlement_id`).

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

auth = pd.read_csv('auth_records.csv', parse_dates=['auth_timestamp'])
clr  = pd.read_csv('clearing_records.csv', parse_dates=['clearing_timestamp'])
sett = pd.read_csv('settlement_records.csv', parse_dates=['settlement_date', 'transaction_date'])

print('auth shape:', auth.shape)
print('clearing shape:', clr.shape)
print('settlement shape:', sett.shape)

---
## Section 1 — Authorisation Analysis

> **Concept:** Authorisation is the real-time gate. Only ~85% of attempts are approved. Declines and timeouts never reach clearing — so your clearing dataset is already a filtered view of reality.

### Q1 · What is the overall authorisation approval rate?
Break it down by `auth_response` (APPROVED / DECLINED / TIMEOUT) and show counts and percentages.

**What to think about:** Why does a data scientist care about the decline rate? What business decisions depend on it?

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q1

In [ ]:
response_counts = auth['auth_response'].value_counts()
response_pct    = auth['auth_response'].value_counts(normalize=True).mul(100).round(1)

summary = pd.DataFrame({'count': response_counts, 'pct': response_pct})
print(summary)
print(f"\nApproval rate: {response_pct.get('APPROVED', 0):.1f}%")

# Insight
print("""
Insight: ~15% of transactions never reach clearing.
If you only analyse clearing data you are missing:
  - Revenue leakage from declines (false positives on fraud models)
  - Retry patterns that can indicate fraud or system issues
  - Merchant conversion rates that affect acquiring relationships
""")

---
### Q2 · Which decline reason code is most common? Which is most costly (by total declined amount)?

**Hint:** Filter to `auth_response == 'DECLINED'`, then group by `response_message`.

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q2

In [ ]:
declines = auth[auth['auth_response'] == 'DECLINED']

by_reason = declines.groupby('response_message').agg(
    count=('auth_id', 'count'),
    total_amount=('auth_amount_gbp', 'sum')
).sort_values('total_amount', ascending=False)

print(by_reason.round(2))

print(f"\nMost frequent reason : {by_reason['count'].idxmax()}")
print(f"Most costly reason   : {by_reason['total_amount'].idxmax()}")
print("""
Insight: 'Insufficient funds' is typically the most common.
'Do not honour' is a catch-all used by issuers for fraud blocks —
it's deliberately vague to prevent card testing attacks.
""")

---
### Q3 · Identify duplicate / retry authorisations

TIMEOUT events represent a terminal that retried after a network failure. Find all `card_last4` + `merchant_id` + date combinations that appear more than once within the same day, and show the time gap between attempts.

**Why it matters:** Duplicate auths can lead to double charges if not detected. Card schemes charge a fee for each auth attempt regardless of outcome.

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q3

In [ ]:
auth['date'] = auth['auth_timestamp'].dt.date

# Find all timeout rows — these are the retry attempts
timeouts = auth[auth['auth_response'] == 'TIMEOUT'].copy()
print(f"Timeout/retry events: {len(timeouts)}")
print(timeouts[['auth_id','auth_timestamp','merchant_name','card_last4','auth_amount_gbp']].head(10))

# Broader check: same card + merchant + date with multiple attempts
auth_sorted = auth.sort_values('auth_timestamp')
dupes = auth_sorted.groupby(['card_last4','merchant_id','date']).filter(lambda g: len(g) > 1)
print(f"\nRows in multi-attempt groups: {len(dupes)}")
print("""
Insight: TIMEOUTs represent the original failed attempt.
The merchant retried and the second attempt was approved.
Risk: if the issuer processed both, the cardholder could be double-charged.
Acquirers use 'retrieval reference numbers' to deduplicate — this is why
settlement reconciliation is non-trivial.
""")

---
## Section 2 — Clearing Analysis

> **Concept:** Clearing is where the real amounts are confirmed and fees are calculated. Not all transactions clear at the authorised amount — restaurants add tips, hotels charge actual stays.

### Q4 · How many cleared transactions have a different amount from the authorised amount?

Join `clearing_records` to `auth_records` on `auth_id`. Calculate the adjustment amount and percentage. Which merchant category (MCC) drives the most adjustments?

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q4

In [ ]:
adjusted = clr[clr['amount_adjusted'] == True].copy()
adjusted['delta_gbp'] = adjusted['clearing_amount_gbp'] - adjusted['auth_amount_gbp']
adjusted['delta_pct'] = (adjusted['delta_gbp'] / adjusted['auth_amount_gbp'] * 100).round(1)

print(f"Adjusted transactions: {len(adjusted)} / {len(clr)} ({len(adjusted)/len(clr):.1%})")
print(f"\nBy adjustment reason:")
print(adjusted.groupby('adjustment_reason')[['delta_gbp']].agg(['count','mean','sum']).round(2))

print(f"\nAvg tip added at restaurants: £{adjusted[adjusted['adjustment_reason']=='tip_added']['delta_gbp'].mean():.2f}")
print(f"Avg hotel delta: £{adjusted[adjusted['adjustment_reason']=='hotel_actual']['delta_gbp'].mean():.2f}")
print("""
Insight: This is a critical data quality issue in payment analytics.
If you use auth amounts for revenue reporting, you over/undercount.
Hotels systematically over-auth (risk management), restaurants under-auth (tips).
Always use clearing_amount_gbp for financial analysis, auth_amount_gbp for fraud analysis.
""")

---
### Q5 · Calculate the effective interchange rate by card type and verify it matches the regulatory caps

Expected: consumer debit ≤ 0.20%, consumer credit ≤ 0.30%, commercial credit uncapped (~1.80%).

**Discussion:** Why do commercial cards have no cap? Who lobbied against capping them?

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q5

In [ ]:
fee_by_card = clr.groupby('card_type').agg(
    total_clearing=('clearing_amount_gbp', 'sum'),
    total_interchange=('interchange_fee_gbp', 'sum'),
    total_scheme=('scheme_fee_gbp', 'sum'),
    total_msc=('msc_total_gbp', 'sum'),
    txn_count=('clearing_id', 'count')
).assign(
    interchange_rate=lambda d: (d['total_interchange'] / d['total_clearing'] * 100).round(3),
    msc_rate=lambda d: (d['total_msc'] / d['total_clearing'] * 100).round(3)
)

print(fee_by_card[['txn_count','total_clearing','interchange_rate','msc_rate']].round(2))

print("""
Regulatory caps (EU/UK IFR):
  Consumer debit:    0.20%  ← should match
  Consumer credit:   0.30%  ← should match
  Commercial credit: uncapped (our dataset uses 1.80%)
  Prepaid:           0.20%

Commercial cards are exempt from IFR caps because they were argued
to serve B2B use cases where the 'consumer protection' rationale
for caps doesn't apply. In practice it means business card users
cost merchants 6x more per transaction.
""")

---
### Q6 · Which merchant receives the least net revenue relative to gross sales (highest effective MSC)?

Calculate net-to-merchant as a percentage of clearing amount per merchant. What drives the difference?

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q6

In [ ]:
by_merchant = clr.groupby(['merchant_id','merchant_name']).agg(
    gross=('clearing_amount_gbp', 'sum'),
    net=('net_to_merchant_gbp', 'sum'),
    interchange=('interchange_fee_gbp', 'sum'),
    txn_count=('clearing_id', 'count')
).assign(
    msc_pct=lambda d: ((d['gross'] - d['net']) / d['gross'] * 100).round(3),
    avg_txn=lambda d: (d['gross'] / d['txn_count']).round(2)
).sort_values('msc_pct', ascending=False)

print(by_merchant[['merchant_name','gross','msc_pct','avg_txn','txn_count']].to_string())

print("""
Insight: Cross-border merchants (Ryanair, Spotify) pay higher scheme fees.
Merchants with high commercial card mix pay more interchange.
This is why large merchants invest heavily in:
  1. Steering customers to debit over credit
  2. Open Banking PIS (zero interchange)
  3. Negotiating interchange++ pricing to see the real breakdown
""")

---
## Section 3 — Settlement Analysis

> **Concept:** Settlement is where money actually moves. It's netted — acquirers and issuers don't transfer per transaction, they calculate the net position across thousands of transactions and transfer once.

### Q7 · What proportion of transactions settle T+1 vs T+2?

Use `settlement_delay_days` in the settlement dataset. Is there a pattern by acquirer or card scheme?

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q7

In [ ]:
delay_dist = sett['settlement_delay_days'].value_counts(normalize=True).mul(100).round(1)
print("Settlement timing:")
print(delay_dist)

by_scheme = sett.groupby(['card_scheme','settlement_delay_days']).size().unstack(fill_value=0)
print("\nBy card scheme:")
print(by_scheme)

by_acquirer = sett.groupby(['acquirer_id','settlement_delay_days']).size().unstack(fill_value=0)
print("\nBy acquirer:")
print(by_acquirer)

print("""
Insight: T+2 settlements create a cash flow gap for merchants.
A merchant processing £100k/day has £200k 'in transit' at any time.
This is why early settlement products (next-day or same-day funding)
are a key acquirer differentiator — merchants pay for faster access to cash.
""")

---
### Q8 · Reconciliation challenge — find cleared transactions that have no settlement record

Join clearing to settlement on `settlement_id`. Any clearing record with a null settlement_id represents a reconciliation break — money that cleared but hasn't settled.

**Real-world context:** This is a P1 incident in any payment operations team.

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q8

In [ ]:
unsettled = clr[clr['settlement_id'].isna()]
print(f"Clearing records with no settlement_id: {len(unsettled)}")

if len(unsettled) > 0:
    print(unsettled[['clearing_id','auth_id','merchant_name','clearing_amount_gbp','clearing_timestamp']].head())
    print(f"Total unsettled amount: £{unsettled['clearing_amount_gbp'].sum():.2f}")

# Also find auth records with no clearing (approved but never cleared)
approved_auths = auth[auth['auth_response'] == 'APPROVED']
cleared_auth_ids = set(clr['auth_id'])
not_cleared = approved_auths[~approved_auths['auth_id'].isin(cleared_auth_ids)]
print(f"\nApproved auths with no clearing record: {len(not_cleared)}")
print(f"Total at-risk amount: £{not_cleared['auth_amount_gbp'].sum():.2f}")
print("""
Insight: Approved-but-not-cleared happens when:
  1. Merchant batch was lost (technical failure)
  2. Merchant voided the transaction after authorising
  3. Timeout between auth and batch close
The held funds on the cardholder account are released after ~7 days
but the merchant never receives payment — a loss.
""")

---
### Q9 · Full pipeline: trace a single transaction from auth → clearing → settlement

Pick the auth_id with the largest adjustment (tip or hotel delta). Show its complete journey across all three datasets, with all relevant timestamps and amounts at each stage.

**This is the core reconciliation skill.** In production you'd do this for every transaction.

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q9

In [ ]:
# Find transaction with largest adjustment
clr_adj = clr[clr['amount_adjusted'] == True].copy()
clr_adj['delta'] = abs(clr_adj['clearing_amount_gbp'] - clr_adj['auth_amount_gbp'])
target_clr = clr_adj.loc[clr_adj['delta'].idxmax()]
target_auth_id = target_clr['auth_id']

# Auth stage
auth_row = auth[auth['auth_id'] == target_auth_id].iloc[0]
# Clearing stage
clr_row  = clr[clr['auth_id'] == target_auth_id].iloc[0]
# Settlement stage
set_row  = sett[sett['settlement_id'] == clr_row['settlement_id']].iloc[0] if pd.notna(clr_row['settlement_id']) else None

print("═" * 60)
print("STAGE 1: AUTHORISATION")
print(f"  auth_id          : {auth_row['auth_id']}")
print(f"  timestamp        : {auth_row['auth_timestamp']}")
print(f"  merchant         : {auth_row['merchant_name']}")
print(f"  card_type        : {auth_row['card_type']}")
print(f"  auth_amount      : £{auth_row['auth_amount_gbp']:.2f}")
print(f"  response         : {auth_row['auth_response']}")
print(f"  money moved?     : NO — hold placed on cardholder account")

print()
print("STAGE 2: CLEARING")
print(f"  clearing_id      : {clr_row['clearing_id']}")
print(f"  timestamp        : {clr_row['clearing_timestamp']}")
print(f"  clearing_amount  : £{clr_row['clearing_amount_gbp']:.2f}")
print(f"  adjustment       : £{clr_row['clearing_amount_gbp'] - clr_row['auth_amount_gbp']:+.2f} ({clr_row['adjustment_reason']})")
print(f"  interchange      : £{clr_row['interchange_fee_gbp']:.4f} ({clr_row['interchange_rate_pct']:.2f}%)")
print(f"  scheme_fee       : £{clr_row['scheme_fee_gbp']:.4f}")
print(f"  processor_fee    : £{clr_row['processor_fee_gbp']:.4f}")
print(f"  MSC total        : £{clr_row['msc_total_gbp']:.4f}")
print(f"  net_to_merchant  : £{clr_row['net_to_merchant_gbp']:.2f}")
print(f"  money moved?     : NO — fees calculated, positions recorded")

if set_row is not None:
    print()
    print("STAGE 3: SETTLEMENT")
    print(f"  settlement_id    : {set_row['settlement_id']}")
    print(f"  settlement_date  : {set_row['settlement_date'].date()}")
    print(f"  delay            : T+{set_row['settlement_delay_days']}")
    print(f"  acquirer         : {set_row['acquirer_id']}")
    print(f"  issuer           : {set_row['issuer_id']}")
    print(f"  batch_txn_count  : {set_row['transaction_count']}")
    print(f"  gross_batch      : £{set_row['gross_amount_gbp']:.2f}")
    print(f"  net_settlement   : £{set_row['net_settlement_gbp']:.2f}")
    print(f"  money moved?     : YES — net transfer between banks")
print("═" * 60)

---
### Q10 · Open-ended — build a daily dashboard

Create a daily summary DataFrame with the following columns for each day in the dataset:

| Column | Source |
|--------|--------|
| `date` | auth_timestamp date |
| `auth_attempts` | count of all auth rows |
| `approval_rate` | APPROVED / total |
| `cleared_txns` | count of clearing rows |
| `gross_cleared_gbp` | sum of clearing_amount_gbp |
| `total_interchange_gbp` | sum of interchange_fee_gbp |
| `net_to_merchants_gbp` | sum of net_to_merchant_gbp |
| `settled_gbp` | sum of settlement gross for that transaction_date |

**Stretch goal:** Plot gross cleared vs net settled over time. What do you notice about the timing gap?

In [ ]:
# YOUR CODE HERE


#### ✅ Answer — Q10

In [ ]:
auth['date'] = auth['auth_timestamp'].dt.date
clr['date']  = clr['clearing_timestamp'].dt.date

daily_auth = auth.groupby('date').agg(
    auth_attempts=('auth_id', 'count'),
    approved=('auth_response', lambda x: (x == 'APPROVED').sum())
).assign(approval_rate=lambda d: (d['approved'] / d['auth_attempts']).round(3))

daily_clr = clr.groupby('date').agg(
    cleared_txns=('clearing_id', 'count'),
    gross_cleared_gbp=('clearing_amount_gbp', 'sum'),
    total_interchange_gbp=('interchange_fee_gbp', 'sum'),
    net_to_merchants_gbp=('net_to_merchant_gbp', 'sum')
).round(2)

daily_set = sett.groupby('transaction_date').agg(
    settled_gbp=('gross_amount_gbp', 'sum')
).round(2)
daily_set.index = daily_set.index.date

dashboard = daily_auth.join(daily_clr, how='outer').join(daily_set, how='outer')
print(dashboard.head(10).to_string())

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
dashboard['gross_cleared_gbp'].plot(ax=ax, label='Gross Cleared', color='steelblue')
dashboard['settled_gbp'].plot(ax=ax, label='Net Settled', color='teal', linestyle='--')
ax.set_title('Daily Gross Cleared vs Net Settled (£)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('£ GBP')
ax.legend()
plt.tight_layout()
plt.show()

print("""
Key observation: Settled amounts are higher than cleared amounts on some days
because settlement batches are NETTED across acquirer-issuer pairs —
one settlement record covers dozens of transactions.
The timing gap (cleared today, settled tomorrow) is the T+1/T+2 lag.
""")

---
## Summary: Key Takeaways

| Stage | Key data fact | DS implication |
|-------|--------------|----------------|
| **Auth** | ~15% of attempts are declined | Clearing data is pre-filtered — always check auth for full picture |
| **Auth** | Amounts can change at clearing | Never use auth_amount for financial reporting |
| **Clearing** | Fees calculated per transaction | Interchange varies 9x between consumer debit and commercial credit |
| **Clearing** | ~1% may never settle | Reconciliation breaks are P1 incidents |
| **Settlement** | Net positions, not per-transaction | Settlement data is aggregated — join back to clearing for detail |
| **Settlement** | T+1 or T+2 | Cash flow modelling requires settlement lag awareness |

**Carry forward:** The `clearing_records.csv` dataset (with `net_to_merchant_gbp` and `interchange_fee_gbp`) will be the input for Lesson 3's merchant profitability analysis.